In [11]:
import os
import time
import requests
import pandas as pd

def scrape_league_matches(api_token: str, season_year: str, output_dir: str):
    """
    Scrapes full match results for targeted training competitions from football-data.org v4
    and outputs a structured tabular machine learning training asset.
    """
    base_url = "https://api.football-data.org/v4/competitions"
    headers = {"X-Auth-Token": api_token} 
    
    target_comps = ["PL", "PD", "BL1", "SA", "FL1", "ELC", "DED", "PPL"]
    
    os.makedirs(output_dir, exist_ok=True)
    all_compiled_matches = []
    
    for comp_code in target_comps:

        endpoint = f"{base_url}/{comp_code}/matches"
        params = {"season": season_year} 
        
        print(f"Querying matches for competition code: {comp_code}...")
        try:
            response = requests.get(endpoint, headers=headers, params=params)
            
            if response.status_code != 200:
                print(f" Failed to fetch {comp_code}: Status {response.status_code}")

                time.sleep(6)
                continue
                
            data = response.json()
            matches_list = data.get("matches", [])
            print(f"Found {len(matches_list)} matches total in API payload.")
            
            for m in matches_list:

                if m.get("status") != "FINISHED":
                    continue
                    
                score = m.get("score", {})
                full_time = score.get("fullTime", {})
                
                home_goals = full_time.get("home")
                away_goals = full_time.get("away")
                
                if home_goals is None or away_goals is None:
                    continue
                    
                home_team = m["homeTeam"]["name"]
                away_team = m["awayTeam"]["name"]
                
                winner_code = score.get("winner") 
                
                if winner_code == "HOME_TEAM":
                    match_result = "HOME_WIN"
                    winner_name = home_team
                    loser_name = away_team
                elif winner_code == "AWAY_TEAM":
                    match_result = "AWAY_WIN"
                    winner_name = away_team
                    loser_name = home_team
                else:
                    match_result = "DRAW"
                    winner_name = "NONE"
                    loser_name = "NONE"
                    
                match_row = {
                    "match_id": m.get("id"),
                    "date": m.get("utcDate"),
                    "competition_code": comp_code,
                    "matchday": m.get("matchday"),
                    "home_team": home_team,
                    "away_team": away_team,
                    "home_goals": int(home_goals),
                    "away_goals": int(away_goals),
                    "goal_difference": int(home_goals) - int(away_goals),
                    "match_result": match_result, 
                    "winner_team": winner_name,
                    "loser_team": loser_name
                }
                all_compiled_matches.append(match_row)
                
            time.sleep(6)
            
        except Exception as e:
            print(f"Parsing error encountered on {comp_code}: {e}")
            continue

    df_matches_all = pd.DataFrame(all_compiled_matches)
    
    output_path = os.path.join(output_dir, f"training_matches_{season_year.replace('/', '_')}.csv")
    df_matches_all.to_csv(output_path, index=False)
    
    print(f"\nExtraction Complete! Compiled {df_matches_all.shape[0]} total finished match training matrices.")
    print(f" File successfully stored at: {output_path}")
    return df_matches_all

if __name__ == "__main__":
    API_TOKEN = "3a4746370f87464388c78623220890ad"
    
    df_training_set = scrape_league_matches(
        api_token=API_TOKEN, 
        season_year='2025', 
        output_dir="D:/FOOTBALL PROJECT/data/raw/matches/"
    )

Querying matches for competition code: PL...
Found 380 matches total in API payload.
Querying matches for competition code: PD...
Found 380 matches total in API payload.
Querying matches for competition code: BL1...
Found 306 matches total in API payload.
Querying matches for competition code: SA...
Found 380 matches total in API payload.
Querying matches for competition code: FL1...
Found 306 matches total in API payload.
Querying matches for competition code: ELC...
Found 557 matches total in API payload.
Querying matches for competition code: DED...
Found 306 matches total in API payload.
Querying matches for competition code: PPL...
Found 306 matches total in API payload.

Extraction Complete! Compiled 2919 total finished match training matrices.
 File successfully stored at: D:/FOOTBALL PROJECT/data/raw/matches/training_matches_2025.csv
